In [1]:
import os
import json
import re
from typing import List, Dict, Any, Tuple
from pathlib import Path
from datetime import datetime

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from groq import Groq
from transformers import pipeline
from dotenv import load_dotenv

load_dotenv()

# ─────────────────────────────────────────────
# CONFIG & MODELS
# ─────────────────────────────────────────────

app = FastAPI(title="Indy ADHD Copilot API")

print("Loading NLP model...")
nlp_classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

GROQ_API_KEY = "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"
client       = Groq(api_key=GROQ_API_KEY)
LLM_MODEL    = "groq/compound-mini"
SESSIONS_DIR = Path("sessions")
SESSIONS_DIR.mkdir(exist_ok=True)

LABEL_DESCRIPTIONS = {
    "affective": "expressing emotions, feelings, distress, sadness, fear, love, anger",
    "cognitive": "thinking, reasoning, reflecting, understanding, analyzing, wondering",
    "agency":    "taking action, making decisions, expressing control, planning, choosing"
}

# ─────────────────────────────────────────────
# SCHEMAS
# ─────────────────────────────────────────────

class ChatRequest(BaseModel):
    session_id: int
    message: str
    history: List[Dict[str, str]]

# ─────────────────────────────────────────────
# NLP LAYER
# ─────────────────────────────────────────────

def score_text(text: str):
    candidate_labels = list(LABEL_DESCRIPTIONS.values())
    result = nlp_classifier(text, candidate_labels, multi_label=False)

    scores = {}
    for label, score in zip(result['labels'], result['scores']):
        for clean_name, desc in LABEL_DESCRIPTIONS.items():
            if label == desc:
                scores[clean_name] = round(score, 4)

    dominant = max(scores, key=scores.get)
    return {"scores": scores, "dominant": dominant}

# ─────────────────────────────────────────────
# SESSION MANAGEMENT
# ─────────────────────────────────────────────

def get_past_context(session_id: int):
    context_blocks = []
    for i in range(session_id - 1, max(0, session_id - 4), -1):
        path = SESSIONS_DIR / f"session_{i:03d}.json"
        if path.exists():
            with open(path, "r") as f:
                s = json.load(f)
                block = f"Session {s['session_id']}: Mood: {s.get('mood')}, Wins: {s.get('small_wins')}"
                context_blocks.append(block)
    return "\n".join(context_blocks) if context_blocks else "No history."


def load_or_create_session(session_id: int) -> dict:
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    if path.exists():
        with open(path, "r") as f:
            return json.load(f)
    return {
        "session_id":       session_id,
        "created":          datetime.now().isoformat(),
        "conversation":     [],
        "cognitive_shifts": [],
        "notes":            {}   # accumulated notes across the session
    }


def save_session(session_id: int, session_data: dict) -> None:
    path = SESSIONS_DIR / f"session_{session_id:03d}.json"
    with open(path, "w") as f:
        json.dump(session_data, f, indent=2)

# ─────────────────────────────────────────────
# NOTE EXTRACTION
# ─────────────────────────────────────────────

def extract_notes_from_response(raw_reply: str) -> Tuple[str, Dict[str, Any]]:
    """
    Extract clean reply text and any notes the AI decided to make.

    The AI will only include ### NOTES if it judged something noteworthy.
    If the message had nothing important, ### NOTES won't appear at all
    and we return an empty dict — no note is created.
    """
    notes: Dict[str, Any] = {}
    clean_text = raw_reply

    if "### NOTES" in raw_reply:
        clean_text = raw_reply.split("### NOTES")[0].strip()
        json_part  = raw_reply.split("### NOTES")[-1].strip()

        try:
            json_part   = re.sub(r"^```(?:json)?", "", json_part).strip()
            json_part   = re.sub(r"```$",          "", json_part).strip()
            json_match  = re.search(r'\{.*\}', json_part, re.DOTALL)

            if json_match:
                notes = json.loads(json_match.group(0))
            else:
                notes = json.loads(json_part)

        except json.JSONDecodeError as e:
            print(f"Warning: Failed to parse notes JSON: {e}")
            # Don't surface malformed notes — just log and move on
            notes = {}

    return clean_text, notes

# ─────────────────────────────────────────────
# ENDPOINT
# ─────────────────────────────────────────────

@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    try:
        # 1. NLP layer — cognitive shift scoring
        analysis = score_text(req.message)

        # 2. RAG — past session context
        past_context = get_past_context(req.session_id)

        # 3. System prompt — AI decides when to make notes
        system_prompt = f"""You are Indy, an ADHD copilot. Be gentle, short, no guilt.
Respond in the same language as the user.

PAST CONTEXT:
{past_context}

---

NOTE-TAKING INSTRUCTIONS:
You have the ability to make notes, but you should only use it when the user says something genuinely worth remembering — such as a personal goal, a fear, a recurring pattern, a childhood memory, a significant relationship, a blocker they keep hitting, or a meaningful win.

Do NOT make notes for:
- Small talk or greetings
- Simple task updates ("I finished the report")
- One-off logistical things
- Anything you'd forget by next session without missing much

If this message IS noteworthy, end your reply with:
### NOTES
{{"title": "short descriptive title", "content": "one or two sentence summary of what matters"}}

If this message is NOT noteworthy, just reply normally. Do not add ### NOTES at all.
Do not mention notes to the user. This is a silent background process.
"""

        # 4. Call Groq
        messages = (
            [{"role": "system", "content": system_prompt}]
            + req.history
            + [{"role": "user", "content": req.message}]
        )

        completion = client.chat.completions.create(
            model=LLM_MODEL,
            messages=messages,
            temperature=0.7,
            max_tokens=1024
        )

        raw_reply = completion.choices[0].message.content

        # 5. Parse reply and notes
        clean_text, notes = extract_notes_from_response(raw_reply)

        # 6. Load session and update
        session_data = load_or_create_session(req.session_id)

        # Append user message
        session_data["conversation"].append({
            "role":      "user",
            "content":   req.message,
            "timestamp": datetime.now().isoformat()
        })

        # Append assistant reply
        session_data["conversation"].append({
            "role":            "assistant",
            "content":         clean_text,
            "timestamp":       datetime.now().isoformat(),
            "cognitive_shift": analysis,
            # Only store notes key if something was actually noted
            **({"notes": notes} if notes else {})
        })

        # Track cognitive shifts
        session_data["cognitive_shifts"].append({
            "message":   req.message,
            "analysis":  analysis,
            "timestamp": datetime.now().isoformat()
        })

        # Accumulate notes under a titled index in the session
        # This builds up a clean record of everything noteworthy across the session
        if notes:
            title = notes.get("title", f"note_{datetime.now().strftime('%H%M%S')}")
            session_data["notes"][title] = {
                "content":   notes.get("content", ""),
                "timestamp": datetime.now().isoformat()
            }
            print(f"  [NOTE] '{title}' added to session notes")
        else:
            print(f"  [NOTE] Nothing noteworthy — no note created")

        save_session(req.session_id, session_data)

        return {
            "reply":           clean_text,
            "cognitive_shift": analysis,
            # Return note if one was made, null if not
            "note_created":    notes if notes else None
        }

    except Exception as e:
        print(f"Error in chat endpoint: {e}")
        raise HTTPException(status_code=500, detail=str(e))

c:\Users\SAGAR\.pyenv\pyenv-win\versions\3.8.10\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading NLP model...


In [2]:
main()

NameError: name 'main' is not defined